In [6]:
from aiida.engine import run_get_node
from topworkchain import PrototypeTopWorkChain
from aiida.orm import FolderData
from aiida import load_profile
load_profile()

Profile<uuid='52dacf545d5c4d089ba5e0dd6267dcf0' name='presto'>

In [2]:
#Run the model
results, node = run_get_node(PrototypeTopWorkChain)
#Load the FolderData node containing the compressed cube files (a folder of files stored on the AiiDA database.)
cube_folder = results["cube_folder"]
#Load the Dict node containing all of the relevant details of the transitions.
transition_info_node = results["transition_info"]
#Convert to a standard Python dict.
transition_info = transition_info_node.get_dict()


04/24/2026 10:13:36 PM <73920> aiida.broker.rabbitmq: [WARNING] RabbitMQ v3.12.1 is not supported and will cause unexpected problems!
04/24/2026 10:13:36 PM <73920> aiida.broker.rabbitmq: [WARNING] It can cause long-running workflows to crash and jobs to be submitted multiple times.
04/24/2026 10:13:36 PM <73920> aiida.broker.rabbitmq: [WARNING] See https://github.com/aiidateam/aiida-core/wiki/RabbitMQ-version-to-use for details.


uuid: b4b44820-ae11-4ee6-949c-189bb720c021 (pk: 29360) (subworkchains.OrcaWorkChain)
uuid: 9680bc50-f2e1-4046-b30a-a0637875f26a (pk: 29386)
uuid: e7097882-3a4f-412c-89ba-bfe4ed614354 (pk: 29400)
uuid: 9ed94971-c190-47c9-990e-ef2b7423b810 (pk: 29415)
uuid: dcc9c294-26cd-450c-b67a-7dbac6414d5d (pk: 29429)
uuid: 09c90336-2965-4558-9a46-e8f735718987 (pk: 29444)
uuid: 8c4edac0-dc59-4d2e-8787-469bb5526c1a (pk: 29458)
uuid: 1e4f1859-8f91-48c9-8cb7-6effb10426e6 (pk: 29472)
uuid: 6908ffb2-7339-465e-8ec9-cfc71f2f0642 (pk: 29486)
uuid: 32203967-8b9a-4810-afb2-cd96676f12bb (pk: 29501)
uuid: 950434fd-b458-4426-b17c-180cd682cda2 (pk: 29515)
uuid: b20b0fd6-a6a7-43c6-83a7-ed3982534345 (pk: 29530)
uuid: 7b9bd7a1-2ff1-4612-b557-a2d97c6eff65 (pk: 29544)
uuid: 73fb8b87-b375-499a-b278-afcbe23b8043 (pk: 29559)
uuid: cee1711f-4f61-4381-9cff-81df9b757000 (pk: 29573)
uuid: c95fd98d-49bc-4ee2-b5bf-9bca7107b27c (pk: 29587)
uuid: 1090c53d-f785-4437-8e12-5ddfb268625e (pk: 29601)
uuid: 109bc670-e3ee-437a-90cf-f3aa1

In [8]:
import nglview as nv
import ipywidgets as widgets
import traitlets as tl
import ase.io
from ase.build import molecule
from pathlib import Path
import shutil
from ipywidgets import Layout

In [10]:
#Delete old cube files.
for entry in (Path(".")).iterdir():
    if entry.is_dir() and entry.name.startswith("extracted_cubes_"):
        shutil.rmtree(entry)

# Create an output directory for the temporary cube files (names after the pk of the calculation)
directory_name = "extracted_cubes_"+str(node.pk)
output_dir = Path(directory_name)
output_dir.mkdir(exist_ok=True)

# Create a list of the cube files in the data node
names = cube_folder.list_object_names()

# Read in the cube files and write them to temporary files
for name in names:
    with cube_folder.open(name, mode="rb") as file_in:
        with open(f"{output_dir}/{name}.cube", "wb") as file_out:
            file_out.write(file_in.read())

#Get a list of excitations.
specific_excitations = list(transition_info.keys())

#Function to change the options available in the transition_dropdown.
def transition_update(change):
    #Isolate specific relevant transitions.
    specific_transitions = transition_info[excitation_dropdown.value]
    #Adjust format of transitions.
    specific_transitions2 = []
    for each_transition in specific_transitions:
        specific_transitions2.append((((each_transition[0])[0]+" -> "+(each_transition[0])[1]+" ("+each_transition[1]+")"), each_transition))
    transition_dropdown.options = specific_transitions2

# Create dropdown widgets for selecting the cube files
excitation_dropdown = widgets.Dropdown(
    options=specific_excitations,
    value=specific_excitations[0] if specific_excitations else None,
    description="Excitation (s)"
 )

transition_dropdown = widgets.Dropdown(
    description="Transition"
 )

#Run transition_update when excitation_dropdown is clicked.
excitation_dropdown.observe(transition_update, names="value")

#Initialise the transition_dropdown with inital options and value.
transition_update({"new": excitation_dropdown.value})
transition_dropdown.value = transition_dropdown.options[0][1] if transition_dropdown.options else None

# Initialize views with current dropdown selections
CUBE_PATH_1 = f"./{directory_name}/{"s"+excitation_dropdown.value+"_"+((transition_dropdown.value[0])[0])[:-1]+".cube"}"
CUBE_PATH_2 = f"./{directory_name}/{"s"+excitation_dropdown.value+"_"+((transition_dropdown.value[0])[1])[:-1]+".cube"}"

# print(CUBE_PATH_1)
# print(CUBE_PATH_2)
# Create NGL views for the selected cube files
molecule1 = nv.show_ase(ase.io.read(CUBE_PATH_1))
molecule2 = nv.show_ase(ase.io.read(CUBE_PATH_2))
NTO_1 = molecule1.add_component(CUBE_PATH_1, ext="cube")
NTO_2 = molecule2.add_component(CUBE_PATH_2, ext="cube")


# Create sliders for adjusting the isovalue (not visible in the NGL view, but used to set the isovalue of the surfaces)
positive_level = widgets.FloatLogSlider(
    value=1e-3,
    base=10,
    min=-5,
    max=-1,
    step=0.1,
    description="+ isovalue",
    readout_format=".1e",
 )
negative_level = widgets.FloatLogSlider(
    value=1e-3,
    base=10,
    min=-5,
    max=-1,
    step=0.1,
    description="- isovalue",
    readout_format=".1e",
 )

# List of colour blind friendly colours
colours = ['#377eb8', '#ff7f00', '#4daf4a',
                  '#f781bf', '#a65628', '#984ea3',
                  '#999999', '#e41a1c', '#dede00']

# To store colour buttons
pos_palette_buttons = []
neg_palette_buttons = []

def set_positive_colour(colour):
    selected_colours["positive"] = colour
    redraw_surfaces()

def set_negative_colour(colour):
    selected_colours["negative"] = colour
    redraw_surfaces()

# Create buttons for each colour for both palettes
for c in colours:
    b = widgets.Button(
        layout=widgets.Layout(width='30px', height='30px'),
        style={'button_color': c}
    )
    b.on_click(lambda _, c=c: set_positive_colour(c))
    pos_palette_buttons.append(b)

for c in colours:
    b = widgets.Button(
        layout=widgets.Layout(width='30px', height='30px'),
        style={'button_color': c}
    )
    b.on_click(lambda _, c=c: set_negative_colour(c))
    neg_palette_buttons.append(b)

pos_palette = widgets.HBox(pos_palette_buttons)
neg_palette = widgets.HBox(neg_palette_buttons)

# Preset colours 
selected_colours = {
    "positive": "#377eb8",
    "negative": "#e41a1c"
}

status = widgets.HTML()

# Define functions to redraw the surfaces when the sliders or colour scheme are changed
def redraw_NTO_1(_=None):
    NTO_1.clear()
    NTO_1.add_surface(
        color=selected_colours["positive"],
        isolevelType="value",
        isolevel=float(positive_level.value),
        opacity=0.5,
    )
    NTO_1.add_surface(
        color=selected_colours["negative"],
        isolevelType="value",
        isolevel=-float(negative_level.value),
        opacity=0.5,
    )
def redraw_NTO_2(_=None):
    NTO_2.clear()
    NTO_2.add_surface(
        color=selected_colours["positive"],
        isolevelType="value",
        isolevel=float(positive_level.value),
        opacity=0.5,
    )
    NTO_2.add_surface(
        color=selected_colours["negative"],
        isolevelType="value",
        isolevel=-float(negative_level.value),
        opacity=0.5,
    )
def redraw_surfaces(_=None):
    redraw_NTO_1()
    redraw_NTO_2()

def update_NTO_1(_=None):
    #print("s"+excitation_dropdown.value+"."+((transition_dropdown.value[0])[0])[:-1]+".cube")
    global NTO_1
    NTO_1.clear()
    NTO_1 = molecule1.add_component(f"./extracted_cubes/{"s"+excitation_dropdown.value+"."+((transition_dropdown.value[0])[0])[:-1]+".cube"}", ext="cube")  # Add path!
    redraw_NTO_1()
    
def update_NTO_2(_=None):
    #print("s"+excitation_dropdown.value+"."+((transition_dropdown.value[0])[1])[:-1]+".cube")
    global NTO_2
    NTO_2.clear()
    NTO_2 = molecule2.add_component(f"./extracted_cubes/{"s"+excitation_dropdown.value+"."+((transition_dropdown.value[0])[1])[:-1]+".cube"}", ext="cube")  # Add path!
    redraw_NTO_2()

# Set up observers to redraw the surfaces when the sliders or colour scheme are changed
for control in [positive_level, negative_level]:
    control.observe(redraw_surfaces, names="value")

excitation_dropdown.observe(update_NTO_1, names="value")
transition_dropdown.observe(update_NTO_1, names="value")
excitation_dropdown.observe(update_NTO_2, names="value")
transition_dropdown.observe(update_NTO_2, names="value")

redraw_surfaces()

# Create a box to contain the colour palettes
palette_box = widgets.VBox([
    widgets.HTML("<b>Positive colour</b>"),
    pos_palette,
    widgets.HTML("<b>Negative colour</b>"),
    neg_palette
    
])

# Create drop down for displaying colours
accordion = widgets.Accordion(children=[palette_box])
accordion.set_title(0, 'Select colours')
accordion.selected_index = None  # start closed
accordion.layout = Layout(width="25%")

# Arrange the widgets in a layout
controls = widgets.VBox(
    [
        widgets.HTML("<b>Cube isosurfaces</b>"),
        accordion,
        status,
    ],
 )
molecule1_box = widgets.VBox([excitation_dropdown, molecule1], layout= Layout(width="100%", flex="1 1 50%", min_width="0"))
molecule2_box = widgets.VBox([transition_dropdown, molecule2], layout= Layout(width="100%", flex="1 1 50%", min_width="0"))
molecule_box = widgets.HBox([molecule1_box, molecule2_box], layout=Layout(width="100%", flex="1 1 100%"))

widgets.VBox([controls, molecule_box], layout = Layout(width="100%", flex="1 1 100%"))
